Checkpoint 1: imports y conexión

In [1]:
import os
import io
import json
from datetime import datetime, timezone

import joblib
import pandas as pd
from sqlalchemy import create_engine, text
from minio import Minio

DB_URL = (
    "postgresql+psycopg://{user}:{pw}@{host}:{port}/{db}".format(
        user=os.environ["DATA_DB_USER"],
        pw=os.environ["DATA_DB_PASSWORD"],
        host=os.environ["DATA_DB_HOST"],
        port=os.environ.get("DATA_DB_PORT", "5432"),
        db=os.environ["DATA_DB_NAME"],
    )
)
engine = create_engine(DB_URL)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar())

PostgreSQL 16.15 (Debian 16.15-1.pgdg13+2) on aarch64-unknown-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


Checkpoint 2: qué versiones hay

In [2]:
versions = pd.read_sql(
    text("SELECT * FROM training.dataset_version ORDER BY created_at DESC"),
    engine,
)
versions

,version,created_at,dag_run_id,row_count,train_fraction,val_fraction,split_seed,notes


Checkpoint 3: elegir versión y cargar

In [ ]:
DATASET_VERSION = versions.iloc[0]["version"]  # or pin one explicitly, e.g. "v1"

df = pd.read_sql(
    text("SELECT * FROM training.covertype_features WHERE dataset_version = :v"),
    engine,
    params={"v": DATASET_VERSION},
)

print(DATASET_VERSION, df.shape)
df["split"].value_counts()

In [ ]:
Checkpoint 4: contrato de features

In [ ]:
TARGET = "cover_type"
CATEGORICAL = ["wilderness_area", "soil_type"]
NUMERIC = [
    "elevation", "aspect", "slope",
    "horizontal_distance_to_hydrology", "vertical_distance_to_hydrology",
    "horizontal_distance_to_roadways",
    "hillshade_9am", "hillshade_noon", "hillshade_3pm",
    "horizontal_distance_to_fire_points",
]
FEATURES = NUMERIC + CATEGORICAL

def xy(split):
    part = df[df["split"] == split]
    return part[FEATURES], part[TARGET]

X_train, y_train = xy("train")
X_val, y_val = xy("validation")
X_test, y_test = xy("test")

y_train.value_counts().sort_index()

In [ ]:
Checkpoint 5: pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

preprocess = ColumnTransformer(
    [
        ("num", "passthrough", NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ]
)

model = Pipeline(
    [
        ("preprocess", preprocess),
        ("clf", RandomForestClassifier(
            n_estimators=200, n_jobs=-1, random_state=42, class_weight="balanced"
        )),
    ]
)

model.fit(X_train, y_train)

Checkpoint 6: evaluación en validation

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

val_pred = model.predict(X_val)
metrics = {
    "val_accuracy": accuracy_score(y_val, val_pred),
    "val_f1_macro": f1_score(y_val, val_pred, average="macro"),
}
print(metrics)
print(classification_report(y_val, val_pred, zero_division=0))

Checkpoint 7: test, una sola vez, al final

In [ ]:
test_pred = model.predict(X_test)
metrics.update(
    test_accuracy=accuracy_score(y_test, test_pred),
    test_f1_macro=f1_score(y_test, test_pred, average="macro"),
)
metrics

Checkpoint 8: publicar en MinIO

In [ ]:
def minio_client():
    endpoint = os.environ.get("MINIO_ENDPOINT", "http://minio:9000")
    return Minio(
        endpoint.split("://", 1)[-1],
        access_key=os.environ["MINIO_ACCESS_KEY"],
        secret_key=os.environ["MINIO_SECRET_KEY"],
        secure=endpoint.startswith("https://"),
    )

MODEL_NAME = "cubierta-forestal"
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
prefix = f"{MODEL_NAME}/{DATASET_VERSION}/{stamp}"

payload = io.BytesIO()
joblib.dump(model, payload)
payload.seek(0)

metadata = {
    "model_name": MODEL_NAME,
    "dataset_version": DATASET_VERSION,
    "trained_at": stamp,
    "features": FEATURES,
    "categorical": CATEGORICAL,
    "target": TARGET,
    "classes": sorted(int(c) for c in model.classes_),
    "metrics": metrics,
    "estimator": "RandomForestClassifier",
}
meta_bytes = json.dumps(metadata, indent=2).encode()

client = minio_client()
bucket = os.environ.get("MODELS_BUCKET", "models")
client.put_object(bucket, f"{prefix}/model.joblib", payload, len(payload.getvalue()))
client.put_object(bucket, f"{prefix}/metadata.json", io.BytesIO(meta_bytes), len(meta_bytes))

print(f"published s3://{bucket}/{prefix}/")

Checkpoint 9: smoke test del artefacto

In [ ]:
obj = client.get_object(bucket, f"{prefix}/model.joblib")
loaded = joblib.load(io.BytesIO(obj.read()))
obj.close(); obj.release_conn()

sample = X_test.head(3)
print(loaded.predict(sample), y_test.head(3).tolist())